# 🦀 Day 5: Async Channels

---

## What are Async Channels?

Channels let async tasks communicate by sending and receiving messages. Tokio provides several channel types for different patterns:

| Channel | Producers | Consumers | Use Case |
|---------|-----------|------------|----------|
| `mpsc` | Many | One | Task coordination, work queues |
| `broadcast` | Many | Many | Event broadcasting |
| `watch` | One | Many | Latest value propagation |
| `oneshot` | One | One | Single response |

In [ ]:
:dep tokio = { version = "1", features = ["full"] }

use tokio::sync::mpsc;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    // Bounded channel: capacity 10, provides backpressure
    let (tx, mut rx) = mpsc::channel::<i32>(10);

    tokio::spawn(async move {
        tx.send(1).await.unwrap();
        tx.send(2).await.unwrap();
        tx.send(3).await.unwrap();
    });

    println!("Received: {}", rx.recv().await.unwrap());
    println!("Received: {}", rx.recv().await.unwrap());
    println!("Received: {}", rx.recv().await.unwrap());
});

## Bounded vs Unbounded Channels

- **Bounded** (`mpsc::channel(n)`): Sender blocks when buffer is full → **backpressure**
- **Unbounded** (`mpsc::unbounded_channel()`): Never blocks on send, but can grow memory

In [ ]:
use tokio::sync::mpsc;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let (tx, mut rx) = mpsc::unbounded_channel::<&str>();

    tx.send("hello").unwrap();  // unbounded: no .await needed
    tx.send("world").unwrap();

    println!("{}", rx.recv().await.unwrap());
    println!("{}", rx.recv().await.unwrap());
});

## tokio::sync::broadcast — Multi-Producer, Multi-Consumer

Every message goes to **all** subscribers. Late joiners miss old messages.

In [ ]:
use tokio::sync::broadcast;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let (tx, _) = broadcast::channel::<String>(16);

    let mut rx1 = tx.subscribe();
    let mut rx2 = tx.subscribe();

    tx.send("event 1".to_string()).unwrap();
    tx.send("event 2".to_string()).unwrap();

    println!("rx1: {}", rx1.recv().await.unwrap());
    println!("rx2: {}", rx2.recv().await.unwrap());
    println!("rx1: {}", rx1.recv().await.unwrap());
    println!("rx2: {}", rx2.recv().await.unwrap());
});

## tokio::sync::watch — Single-Producer, Multi-Consumer (Latest Value)

Subscribers always get the **latest** value. Old values are overwritten.

In [ ]:
use tokio::sync::watch;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let (tx, mut rx) = watch::channel(0i32);

    let mut rx2 = rx.clone();

    tx.send(1).unwrap();
    tx.send(2).unwrap();
    tx.send(3).unwrap();

    assert_eq!(*rx.borrow(), 3);
    assert_eq!(*rx2.borrow(), 3);
    println!("Latest value: {}", *rx.borrow());
});

## tokio::sync::oneshot — Single-Use Channel

Perfect for request-response: one sender, one receiver, one message.

In [ ]:
use tokio::sync::oneshot;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let (tx, rx) = oneshot::channel::<i32>();

    tokio::spawn(async move {
        tx.send(42).unwrap();
    });

    let result = rx.await.unwrap();
    println!("Received: {}", result);
});

## Practical Pattern: Task Coordination

Use `mpsc` to coordinate a producer and consumer.

In [ ]:
use tokio::sync::mpsc;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let (tx, mut rx) = mpsc::channel(5);

    let producer = tokio::spawn(async move {
        for i in 1..=5 {
            tx.send(i * 10).await.unwrap();
        }
    });

    let mut sum = 0i32;
    while let Some(msg) = rx.recv().await {
        sum += msg;
    }
    producer.await.unwrap();
    println!("Sum: {}", sum);
});

## 🏋️ Exercise: Build an Async Pub-Sub System

Use `broadcast` to simulate a pub-sub system. Create a channel, spawn 3 subscriber tasks that each print received messages, and have the main task send 3 events. Each subscriber should receive all 3.

In [ ]:
use tokio::sync::broadcast;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    // YOUR CODE HERE
    // 1. Create broadcast channel
    // 2. Spawn 3 tasks that subscribe and print messages
    // 3. Send 3 events from main
    // 4. Use tokio::time::sleep to let subscribers receive
});

## 🎯 Key Takeaways

1. **mpsc** — multi-producer, single-consumer; bounded = backpressure, unbounded = no blocking
2. **broadcast** — multi-producer, multi-consumer; every subscriber gets every message
3. **watch** — single-producer, multi-consumer; only latest value
4. **oneshot** — one message, one sender, one receiver
5. Use `Runtime::new().unwrap().block_on(async { ... })` to run async code in EvCxR

---

**Tomorrow:** Streams — async iterators with tokio-stream 🌊